In [ ]:
%sql
-- silver prices: 15/60 min -> hourly average (market coupling runs on 15-min periods since October 2025)
USE CATALOG energy;

CREATE OR REPLACE TABLE d2_silver.silver_prices_hourly
USING DELTA
AS
SELECT
  zone_code,
  date_trunc('HOUR', ts_utc) AS ts_utc,
  AVG(price_eur_mwh) AS price_eur_mwh,
  COUNT(*) AS n_points
FROM d2_bronze.bronze_prices
GROUP BY zone_code, date_trunc('HOUR', ts_utc);

In [ ]:
%sql
-- silver net positions
CREATE OR REPLACE TABLE d2_silver.silver_net_position_hourly
USING DELTA
AS
SELECT
  zone_code,
  date_trunc('HOUR', ts_utc) AS ts_utc,
  AVG(net_position_mw) AS net_position_mwh,
  COUNT(*) AS n_points
FROM d2_bronze.bronze_net_position
GROUP BY zone_code, date_trunc('HOUR', ts_utc);

In [ ]:
%sql
-- silver physical flows
CREATE OR REPLACE TABLE d2_silver.silver_flows_hourly
USING DELTA
AS
SELECT
  from_zone, to_zone,
  date_trunc('HOUR', ts_utc) AS ts_utc,
  AVG(flow_mw) AS flow_mwh,
  COUNT(*) AS n_points
FROM d2_bronze.bronze_flows
GROUP BY from_zone, to_zone, date_trunc('HOUR', ts_utc);

In [ ]:
%sql
-- silver scheduled exchanges
CREATE OR REPLACE TABLE d2_silver.silver_scheduled_hourly
USING DELTA
AS
SELECT
  from_zone, to_zone,
  date_trunc('HOUR', ts_utc) AS ts_utc,
  AVG(scheduled_mw) AS sched_mwh,
  COUNT(*) AS n_points
FROM d2_bronze.bronze_scheduled
GROUP BY from_zone, to_zone, date_trunc('HOUR', ts_utc);

In [ ]:
%sql
-- hours with a price per zone (all zones should have the same number)
SELECT zone_code, COUNT(*) AS n_hours, MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts
FROM d2_silver.silver_prices_hourly
GROUP BY zone_code
ORDER BY zone_code;